<a href="https://colab.research.google.com/github/AnnaShtyn/python_for_hw_tasks/blob/main/Anna_Shtyn_HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [11]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
from math import ceil

%matplotlib inline

effect_size = sms.proportion_effectsize(0.2, 0.19)

req_n = sms.NormalIndPower().solve_power(
    effect_size,
    power= 0.8,
    alpha=0.05,
    ratio=1
)
req_n = ceil(req_n)
req_n

24638

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [27]:
df = pd.read_csv('drive/MyDrive/cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [34]:
mean_ret_g30 = df[df['version'] == 'gate_30']['retention_7'].mean()
mean_ret_g40 = df[df['version'] == 'gate_40']['retention_7'].mean()

print(mean_ret_g30, mean_ret_g40)

# З середніх значень у популяції бачимо, що версія gate_40 дає менший % повернення
#  Отже, формулюємо гіпотези: Но: р=0,19; На: р<0,19.

0.19020134228187918 0.18200004396667327


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [54]:
# Сформуємо вибірки для тестування
contr_sample = df[df['version'] == 'gate_30'].sample(n=req_n, random_state=22)
treatm_sample = df[df['version'] == 'gate_40'].sample(n=req_n, random_state=22)

# З'єднаємо вибірки для проведення тестування
ab_test = pd.concat([contr_sample, treatm_sample], axis=0)
ab_test.reset_index(drop=True, inplace=True)

# Проводимо z-тестування
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

contr_res = ab_test[ab_test['version'] == 'gate_30']['retention_7']
treatm_res = ab_test[ab_test['version'] == 'gate_40']['retention_7']

n_contr = contr_res.count()
n_treatm = treatm_res.count()
successes = [contr_res.sum(), treatm_res.sum()]
nobs = [n_contr, n_treatm]

z_stat, pval = proportions_ztest(successes, nobs=nobs)
(l_contr, l_treatm), (u_contr, u_treatm) = proportion_confint(
    successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat}')
print(f'p-value: {pval}')
print(f'Довірчий інтервал 95% для групи control: [{l_contr}, {u_contr}]')
print(f'Довірчий інтервал 95% для групи treatment: [{l_treatm}, {u_treatm}]')

#  1.Оскільки p-value < alpha (0.0014 < 0.05), вважаємо результати статистично значущими, тому відхиляємо нульову гіпотезу.
#  2.Інтервали не перетинаються, тобто ми пересвідчуємось у статистичній значущості результатів.

z statistic: 3.20011597372952
p-value: 0.0013737229952302259
Довірчий інтервал 95% для групи control: [0.1842487285867723, 0.19402872899907073]
Довірчий інтервал 95% для групи treatment: [0.17320105270117436, 0.18275316436189892]


4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [61]:
#  Гіпотези: Но: утримання клієнта однакове для всіх версій; На: утримання клієнта відрізняється залежно від версії.

crosstab = pd.crosstab(ab_test['version'], ab_test['retention_7'])

chi2, p, dof, expected = stats.chi2_contingency(crosstab)
print(f'p-value = {p}\n')

# Отже, оскільки p-value значно менше від alpha (0.0014 < 0.05), можемо однозначно
#  констатувати, що зміна часу появи "воріт" у грі впливає на утримання гравця
#  на 7 день, оскільки є мізерна ймовірність випадково отримати такі ж результати
#  для обох гіпотез.

p-value = 0.0014302330491884595

